# This notebook explores the performance of the Hybrid CNN-LSTM, CNN-LSTM-RESIDUAL and CNN-CONFORMER models on a classification task using EEG data.

In [ ]:
!pip install keras==3.13.2

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import keras
from keras.models import Sequential
from keras.layers import Dense, Activation, Flatten, Dropout, Conv2D, LSTM, BatchNormalization, MaxPooling2D, Reshape
from keras.utils import to_categorical
from keras.callbacks import EarlyStopping, ModelCheckpoint

SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
keras.utils.set_random_seed(SEED)


In [ ]:
IN_COLAB = "COLAB_GPU" in os.environ

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    base_path = "/content/drive/MyDrive/Colab Notebooks/cvae_project/"
    os.chdir(base_path)
else:
    base_path = "."

In [ ]:
# Importing custom models and preprocessing
from src.preprocessing import data_prep as dp

from src.models import cnn_lstm, cnn_lstm_res, cnn_conformer

# # In case of reloading...
# import importlib
# importlib.reload(cnn_lstm)
# importlib.reload(cnn_lstm_res)
# importlib.reload(cnn_conformer)

## (i) Load and pre-processing the dataset; prepare the training, validation, and test data.



In [ ]:
# ----------------------------
# Loading Data
# ----------------------------
X_test = np.load(base_path + "data/X_test.npy")
y_test = np.load(base_path + "data/y_test.npy")
person_train_valid = np.load(base_path +"data/person_train_valid.npy")
X_train_valid = np.load(base_path +"data/X_train_valid.npy")
y_train_valid = np.load(base_path +"data/y_train_valid.npy")
person_test = np.load(base_path +"data/person_test.npy")

# ----------------------------
#  Data Pre-Processing - dp.prepare_dataset (mode = 'cnn') for CNN_LSTM, CNN_LSTM_RESIDUAL
# ----------------------------
y_train_valid -= 769
y_test -= 769

x_train, x_valid, x_test, y_train, y_valid, y_test = dp.prepare_dataset(
    X_train_valid, y_train_valid, X_test, y_test, mode='cnn'
)

## (ii) CNN-LSTM: Defining the model architecture, 50 epoch run, visualization, and performance.


In [ ]:
model = cnn_lstm.CNN_LSTM()
model.build()
model.summary()

In [ ]:
# ----------------------------
# CNN LSTM
# ----------------------------
model = cnn_lstm.CNN_LSTM(input_shape=(250, 1, 22))
learning_rate = 1e-3
model.compile(loss='categorical_crossentropy',
                  optimizer=keras.optimizers.Adam(learning_rate),
                  metrics=['accuracy'])
#model.build()


In [ ]:
cnn_lstm_results = model.fit(x_train,
             y_train,
             batch_size=64,
             epochs=50,
             validation_data=(x_valid, y_valid), verbose=True)


In [ ]:
# Plotting accuracy trajectory
plt.plot(cnn_lstm_results.history['accuracy'])
plt.plot(cnn_lstm_results.history['val_accuracy'])
plt.title('CNN-LSTM model accuracy trajectory')
plt.ylabel('accuracy')
plt.xlabel('epoch')
plt.legend(['train', 'val'], loc='upper left')
plt.show()

# Plotting loss trajectory
plt.plot(cnn_lstm_results.history['loss'],'o')
plt.plot(cnn_lstm_results.history['val_loss'],'o')
plt.title('CNN-LSTM model loss trajectory')
plt.ylabel('loss')
plt.xlabel('epoch')
plt.legend(['train', 'val'], loc='upper left')
plt.show()

In [ ]:

cnn_lstm_score = model.evaluate(x_test, y_test, verbose=0)
print('Test accuracy of the hybrid CNN-LSTM model:',cnn_lstm_score[1])

## (iii) CNN-LSTM-RESIDUAL: Defining the model architecture, 50 epoch run, visualization, and performance.

In [ ]:
model = cnn_lstm_res.CNN_LSTM_RES()
dummy = np.zeros((1, 250, 1, 22))  # batch of 1, matching your input shape
model(dummy)                        # triggers call(), builds all branches
model.summary(expand_nested=True)

In [ ]:
# ----------------------------
#  CNN LSTM RESIDUAL
# ----------------------------
model = cnn_lstm_res.CNN_LSTM_RES(input_shape=(250, 1, 22))
learning_rate = 1e-3
model.compile(loss='categorical_crossentropy',
                  optimizer=keras.optimizers.Adam(learning_rate),
                  metrics=['accuracy'])
model.build(input_shape=(250, 1, 22))


In [ ]:
cnn_lstm_residual_results = model.fit(x_train,
             y_train,
             batch_size=64,
             epochs=50,
             validation_data=(x_valid, y_valid), verbose=True)

model.evaluate(x_test, y_test, verbose=0)

In [ ]:
# Plotting accuracy trajectory
plt.plot(cnn_lstm_residual_results.history['accuracy'])
plt.plot(cnn_lstm_residual_results.history['val_accuracy'])
plt.title('CNN-LSTM RESIDUAL model accuracy trajectory')
plt.ylabel('accuracy')
plt.xlabel('epoch')
plt.legend(['train', 'val'], loc='upper left')
plt.show()

# Plotting loss trajectory
plt.plot(cnn_lstm_residual_results.history['loss'],'o')
plt.plot(cnn_lstm_residual_results.history['val_loss'],'o')
plt.title('CNN-LSTM RESIDUAL model loss trajectory')
plt.ylabel('loss')
plt.xlabel('epoch')
plt.legend(['train', 'val'], loc='upper left')
plt.show()

## (iv) CNN-CONFORMER: Defining the model architecture, 50 epoch run, visualization, and performance.

In [ ]:
model = cnn_conformer.CNN_CONFORMER()
dummy = np.zeros((1, 250, 1, 22))
model(dummy)
model.summary(expand_nested=True)

In [ ]:
# ----------------------------
#  CNN Conformer
# ----------------------------

model = cnn_conformer.CNN_CONFORMER(input_shape=(250, 1, 22))
learning_rate = 1e-3
model.compile(loss='categorical_crossentropy',
                  optimizer=keras.optimizers.Adam(learning_rate),
                  metrics=['accuracy'])
model.build(input_shape=(250, 1, 22))

In [ ]:
cnn_conf_results = model.fit(x_train,
             y_train,
             batch_size=64,
             epochs=50,
             validation_data=(x_valid, y_valid), verbose=True)

model.evaluate(x_test, y_test, verbose=0)

In [ ]:
# Plotting accuracy trajectory
plt.plot(cnn_conf_results.history['accuracy'])
plt.plot(cnn_conf_results.history['val_accuracy'])
plt.title('CNN Conformer model accuracy trajectory')
plt.ylabel('accuracy')
plt.xlabel('epoch')
plt.legend(['train', 'val'], loc='upper left')
plt.show()

# Plotting loss trajectory
plt.plot(cnn_conf_results.history['loss'],'o')
plt.plot(cnn_conf_results.history['val_loss'],'o')
plt.title('CNN-Conformer model loss trajectory')
plt.ylabel('loss')
plt.xlabel('epoch')
plt.legend(['train', 'val'], loc='upper left')
plt.show()

# Results & Discussion

## Test Results Summary

| Model             | Accuracy |
|-------------------|----------|
| CNN-LSTM          | ~62.2%   |
| CNN-LSTM-Residual | 58.41%   |
| CNN-Conformer     | 56.77%   |

*CNN-LSTM accuracy is reported as a mean across two runs (65%, 59.31%)
reflecting runtime variability observed in Google Colab.*

## Discussion

In individual runs, CNN-LSTM achieved the highest mean test accuracy at ~62.2%, followed by CNN-LSTM-Residual at 58.41% and CNN-Conformer at 56.77%. The three models perform within a range of approximately 5%, suggesting the CNN feature extraction frontend is the dominant contributor to performance regardless of the temporal modelling strategy used in the tail.

The CNN-LSTM-Residual performing comparably to the plain CNN-LSTM is notable.  The residual connections do not appear to be providing a clear benefit in this setting. This may warrant further investigation into the learning rate, dropout rates.
The CNN-Conformer ranking last in individual runs is surprising given that attention-based architectures have shown strong results on EEG motor imagery tasks in the literature. With only 50 epochs and a sequence length of 4 tokens after the CNN frontend, the Conformer may not have enough temporal context to fully leverage self-attention.

 A standalone baseline result is documented in `00_baseline_cnn_lstm.ipynb`.

## Future Work

All three models show signs of overfitting, suggesting the following adjustments:

- **CNN-LSTM**: Increase training budget beyond 50 epochs; val loss plateaus
  rather than diverging, suggesting the model has room to improve with longer training.
- **CNN-LSTM-Residual**: Increase dropout and add L2 regularization to conv layers; val loss diverges from epoch 10 indicating the model has too much capacity relative to the dataset.
- **CNN-Conformer**: Reduce model capacity (d_model 128 -> 64, 2 blocks -> 1) before tuning regularization.

